In [0]:
from pyspark.sql import functions as F

sales = spark.table("retailnew.silver.pos_clean")
dim_prod = spark.table("retailnew.gold.dim_product").filter("current_flag = true")
dim_cust = spark.table("retailnew.gold.dim_customer").filter("current_flag = true")

fact = (
    sales
    .join(dim_prod.select("product_id"), "product_id", "left")
    .join(dim_cust.select("customer_id"), "customer_id", "left")
    .withColumn("sales_amount", F.col("amount"))
)

fact.show()

fact = fact.select("transaction_id", "customer_id", "product_id", "qty", "amount", "transaction_date", "store_id", "sales_amount")

(
    fact.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("retailnew.gold.fact_sales")
)

